 **Homework 4:**

**Problem # 1**

**Automating Daily Sales and Performance Reports**:
Many organizations still rely on staff to open raw sales exports every morning, total the numbers manually, and reformat them into a manager-friendly report. This takes time, repeats the same steps every day, and increases the chance of formula or copy-paste errors. In this assignment, students build a small reporting workflow in Python that reads daily transaction data, calculates business metrics, and produces a polished summary that could realistically support a sales or operations meeting.
**Objective**
Use Python in Google Colab to import a daily sales file, calculate basic performance measures, and generate a clean business summary report.
**Expected outputs and deliverables**
• Total sales revenue for the dataset
• One Google Colab notebook with commented Python code
• Total units sold
• A loaded and reviewed dataset using pandas
• Sales by category
• At least three summary calculations
• Sales by region
• At least one grouped report
• Top-performing products
• A short business interpretation of the results
• A clean summary table or printed management report






In [17]:
import pandas as pd
from google.colab import auth
import gspread
from google.auth import default

# --- STEP 1: DATA LOADING & REVIEW ---
# 1. Authorize and connect to the Google Sheet
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Open the sheet and load data into a pandas DataFrame for the "Review" requirement
spreadsheet_name = "Daily Sales Data April 2026"
worksheet = gc.open(spreadsheet_name).sheet1
df = pd.DataFrame(worksheet.get_all_records())

# Review the dataset (First requirement)
print("--- DATASET REVIEW (First 5 Rows) ---")
print(df.head())
print("-" * 45)

# Convert the DataFrame back to a list of dictionaries for beginner-friendly processing
sales_data = df.to_dict('records')

# --- STEP 2: SUMMARY CALCULATIONS ---
# Initializing variables
total_revenue = 0
total_units = 0
category_totals = {}
region_totals = {}
product_totals = {}
i = 0

# Using a WHILE loop to demonstrate flow control (Requirement)
while i < len(sales_data):
    item = sales_data[i]

    # 1. IF statement and CONTINUE to handle errors (Requirement)
    if item["Units_Sold"] <= 0:
        i += 1
        continue

    # 2. Basic Calculations
    line_revenue = item["Units_Sold"] * item["Unit_Price"]
    total_revenue += line_revenue
    total_units += item["Units_Sold"]

    # 3. Grouping Logic (Calculates Sales by Category and Region)
    cat = item["Category"]
    reg = item["Region"]
    prod = item["Product"]

    category_totals[cat] = category_totals.get(cat, 0) + line_revenue
    region_totals[reg] = region_totals.get(reg, 0) + line_revenue
    product_totals[prod] = product_totals.get(prod, 0) + line_revenue

    # BREAK statement safety check (Requirement)
    if total_revenue > 500000:
        break

    i += 1

# Find Top Performing Product
top_product = max(product_totals, key=product_totals.get)

# --- STEP 3: PRINTED MANAGEMENT REPORT ---
print("\n" + "="*45)
print("       DAILY PERFORMANCE SUMMARY REPORT")
print("="*45)
print(f"Total Revenue:         ${total_revenue:,.2f}")
print(f"Total Units Sold:      {total_units}")
print(f"Top Product:           {top_product}")
print("-" * 45)

# Grouped Report: Sales by Category
print(f"{'CATEGORY':<20} | {'REVENUE':>20}")
for category, amount in category_totals.items():
    print(f"{category:<20} | ${amount:>19,.2f}")

print("-" * 45)

# Grouped Report: Sales by Region
print(f"{'REGION':<20} | {'REVENUE':>20}")
for region, amount in region_totals.items():
    print(f"{region:<20} | ${amount:>19,.2f}")
print("="*45)

# --- STEP 4: BUSINESS INTERPRETATION ---
print("\n[MANAGEMENT INTERPRETATION]")
print(f"1. Revenue Goals: Total sales reached ${total_revenue:,.2f}. This represents a stable daily intake.")
print(f"2. Product Focus: {top_product} is our primary driver of value today.")
print(f"3. Regional Insight: The {max(region_totals, key=region_totals.get)} region is currently leading in total sales volume.")
if total_revenue > 15000:
    print("4. Action: Performance is high. No immediate changes to strategy required.")
else:
    print("4. Action: Review low-performing regions to identify potential market gaps.")

--- DATASET REVIEW (First 5 Rows) ---
         Date   Product     Category  Units_Sold  Unit_Price Region
0  2026-04-15    Laptop  Electronics           5        1000  North
1  2026-04-15     Mouse  Accessories          25          20  North
2  2026-04-15   Monitor  Electronics          10         200  South
3  2026-04-15  Keyboard  Accessories           0          45   West
4  2026-04-15     Phone  Electronics           8         800   East
---------------------------------------------

       DAILY PERFORMANCE SUMMARY REPORT
Total Revenue:         $18,850.00
Total Units Sold:      101
Top Product:           Phone
---------------------------------------------
CATEGORY             |              REVENUE
Electronics          | $          15,600.00
Accessories          | $           2,500.00
Furniture            | $             750.00
---------------------------------------------
REGION               |              REVENUE
North                | $           6,100.00
South                

**Problem # 2**

**Tracking Low Inventory and Reorder Needs:**
Inventory shortages can delay customer orders, create lost sales, and frustrate staff. In many workplaces, someone still checks stock levels manually in spreadsheets and decides what needs to be reordered. This project asks students to automate that review by identifying items below their reorder threshold and creating a report that a manager could use to trigger replenishment.
**Objective**
Build a Python workflow that reads an inventory file, detects low-stock items, and creates a reorder alert report.
**Expected outputs and Colab deliverables**
• A list of items below reorder level	• One Colab notebook with clear logic and comments
• A count of low-stock items	• A filtered low-stock report
• A sorted reorder report from most urgent to least urgent	• At least one sorted output table
• Optional urgent alerts for critically low items	• A short explanation of how the automation supports inventory control


In [18]:
import pandas as pd
from google.colab import auth, userdata
import gspread
from google.auth import default

# 1. SETUP: Authorization and Loading
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Open the new Inventory sheet
spreadsheet_name = "Inventory Management Data April 2026"
worksheet = gc.open(spreadsheet_name).sheet1
inventory_list = worksheet.get_all_records()

# 2. INITIALIZE TRACKERS
reorder_queue = []
urgent_alerts = 0
total_items_checked = 0

# 3. PROCESSING LOGIC
# Using a while loop to demonstrate flow control
while total_items_checked < len(inventory_list):
    item = inventory_list[total_items_checked]

    # Simple calculation: How many units are we missing?
    stock_diff = item["Reorder_Level"] - item["Current_Stock"]

    # IF statement: Check if stock is low
    if item["Current_Stock"] < item["Reorder_Level"]:

        # Nested IF: Check for "Critical" levels (less than 20% of reorder level)
        is_urgent = False
        if item["Current_Stock"] <= (item["Reorder_Level"] * 0.2):
            is_urgent = True
            urgent_alerts += 1

        # Add to our list for the final report
        # We store the 'stock_diff' so we know how much to buy
        reorder_queue.append({
            "Name": item["Item_Name"],
            "Need": stock_diff,
            "Supplier": item["Supplier"],
            "Urgent": is_urgent
        })
    else:
        # CONTINUE: Skip items that have plenty of stock
        total_items_checked += 1
        continue

    # BREAK: Safety check (if our queue gets too long for one purchase order)
    if len(reorder_queue) >= 50:
        print("Warning: Purchase order capacity reached.")
        break

    total_items_checked += 1

# 4. SORTING: Sort the report by most urgent (Need) to least urgent
# This creates the "Sorted Reorder Report" requirement
reorder_queue.sort(key=lambda x: x['Need'], reverse=True)

# 5. GENERATE THE MANAGEMENT REPORT
print("="*55)
print("          INVENTORY REPLENISHMENT REPORT")
print("="*55)
print(f"{'ITEM NAME':<20} | {'ORDER QTY':<10} | {'STATUS'}")
print("-" * 55)

for entry in reorder_queue:
    status = "!!! CRITICAL !!!" if entry["Urgent"] else "Low Stock"
    print(f"{entry['Name']:<20} | {entry['Need']:<10} | {status}")

print("-" * 55)
print(f"Total Items to Reorder: {len(reorder_queue)}")
print(f"Critical Alerts:        {urgent_alerts}")
print("="*55)

# 6. BUSINESS INTERPRETATION
print("\n[INVENTORY CONTROL ANALYSIS]")
print(f"Our automation scanned {total_items_checked} unique SKUs.")
print(f"We identified {len(reorder_queue)} items that have fallen below safety stock levels.")
print("Automating this prevents 'Stockouts,' which lead to lost sales and customer")
print("dissatisfaction. By prioritizing the 'Critical' items, the warehouse")
print("team can focus on the most high-risk shortages first.")

          INVENTORY REPLENISHMENT REPORT
ITEM NAME            | ORDER QTY  | STATUS
-------------------------------------------------------
Puppy Training Pads  | 17         | Low Stock
Lavender Shampoo     | 15         | Low Stock
Nylon Leash          | 8          | !!! CRITICAL !!!
Ear Cleaner          | 8          | Low Stock
Travel Crate         | 4          | !!! CRITICAL !!!
Grooming Brush       | 2          | Low Stock
-------------------------------------------------------
Total Items to Reorder: 6
Critical Alerts:        2

[INVENTORY CONTROL ANALYSIS]
Our automation scanned 10 unique SKUs.
We identified 6 items that have fallen below safety stock levels.
Automating this prevents 'Stockouts,' which lead to lost sales and customer
dissatisfaction. By prioritizing the 'Critical' items, the warehouse
team can focus on the most high-risk shortages first.


In [2]:
!pip install -q -U google-generativeai

In [9]:
import pandas as pd
import google.generativeai as genai
from google.colab import auth, userdata
import gspread
from google.auth import default

# 1. AUTHENTICATION & API KEY ACCESS
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Access your GEMINI_KEY from the Colab Secrets panel
api_key = userdata.get('GEMINI_KEY')
genai.configure(api_key=api_key)
model = genai.GenerativeModel('models/gemini-2.5-flash')

# 2. LOAD DATA
spreadsheet_name = "Inventory Management Data April 2026"
worksheet = gc.open(spreadsheet_name).sheet1
inventory_data = worksheet.get_all_records()

# 3. CORE LOGIC: IDENTIFY LOW STOCK
reorder_items = []
i = 0

while i < len(inventory_data):
    row = inventory_data[i]
    if row["Current_Stock"] < row["Reorder_Level"]:
        # Calculate the shortage
        shortage = row["Reorder_Level"] - row["Current_Stock"]
        reorder_items.append(f"Product: {row['Item_Name']}, Category: {row['Category']}, Missing: {shortage} units")

    i += 1

# 4. TASK FOR THE API (The "Prompt")
# We turn our list into a single string to send to Gemini
inventory_string = "\n".join(reorder_items)

prompt = f"""
You are a Business Analytics expert. Look at this low-inventory list for a pet product company:
{inventory_string}

Please provide:
1. A 2-sentence executive summary of the risk.
2. A recommendation on which category to prioritize (Pet Care, Accessories, or Hygiene).
3. One professional sentence we can send to the suppliers.
"""

# 5. EXECUTE API TASK
print("Consulting Gemini AI for business interpretation...")
response = model.generate_content(prompt)

# 6. OUTPUT THE FINAL REPORT
print("\n" + "="*50)
print("        AI-POWERED INVENTORY STRATEGY")
print("="*50)
print(response.text)
print("="*50)

Consulting Gemini AI for business interpretation...

        AI-POWERED INVENTORY STRATEGY
Here is the breakdown of the low-inventory situation:

### 1. Executive Summary of the Risk

The company faces an immediate risk of significant sales loss and customer dissatisfaction due to critical stock shortages across multiple pet product categories. With 54 units missing across six distinct products, prompt replenishment is essential to prevent further revenue erosion and maintain customer loyalty.

### 2. Recommendation for Category Prioritization

Prioritize the **Pet Care** category.
This category has the highest total number of missing units (23 units: 15 for Lavender Shampoo + 8 for Ear Cleaner) across two distinct products, posing the most significant immediate risk to overall sales volume and customer fulfillment compared to Hygiene (17 units) or Accessories (8 units).

### 3. Professional Sentence to Suppliers

"To mitigate immediate sales loss and customer impact from critically lo

**Problem # 3**

**Tracking Task Deadlines and Follow-Ups:**
Professionals often manage many deadlines at once, including project milestones, compliance tasks, and client follow-ups. When everything is tracked manually, it becomes easy to miss an urgent due date. This assignment gives students a practical workflow for reading a task list, identifying what is overdue or coming soon, and creating a reminder summary.
**Objective**
Develop a Python notebook that reads task data, identifies upcoming and overdue work, and generates a reminder report.
**Outputs and deliverables**
• Tasks due within the next seven days
• One Colab notebook with date calculations
• Overdue tasks
• At least two deadline categories such as upcoming and overdue
• Tasks grouped by priority
• A final reminder table
• A reminder summary report
• A short explanation of how this helps project coordination




In [19]:
import pandas as pd
import google.generativeai as genai
from google.colab import auth, userdata
import gspread
from google.auth import default
from datetime import datetime, timedelta

# 1. AUTHENTICATION & DATA LOADING
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Using the stable Spreadsheet ID
spreadsheet_id = "1b7_M7ZJIBbjaTNrUeORMEQemuZRkbrYDb1ddj3knYCY"
worksheet = gc.open_by_key(spreadsheet_id).sheet1
all_tasks = worksheet.get_all_records()

# 2. DATE CALCULATIONS (Deliverable: Date Calculations)
today = datetime(2026, 4, 18)
seven_days_later = today + timedelta(days=7)

# 3. INITIALIZING CATEGORIES
overdue_tasks = []
upcoming_tasks = []
high_priority_tasks = []
medium_priority_tasks = []
low_priority_tasks = []

# 4. DATA PROCESSING LOOP
i = 0
while i < len(all_tasks):
    task = all_tasks[i]
    if task["Status"] == "Completed":
        i += 1
        continue

    # Parse the date
    task_date = datetime.strptime(task["Due_Date"], "%Y-%m-%d")

    # Categorize by Deadline
    if task_date < today:
        overdue_tasks.append(task)
    elif today <= task_date <= seven_days_later:
        upcoming_tasks.append(task)

    # Categorize by Priority
    prio = task["Priority"].upper()
    if prio == "HIGH":
        high_priority_tasks.append(task["Task_Name"])
    elif prio == "MEDIUM":
        medium_priority_tasks.append(task["Task_Name"])
    else:
        low_priority_tasks.append(task["Task_Name"])

    i += 1

# 5. AI SUMMARY REPORT (Using the most stable model name)
try:
    api_key = userdata.get('GEMINI_KEY')
    genai.configure(api_key=api_key)
    # Using 'gemini-2.5-flash' for maximum reliability
    model = genai.GenerativeModel('models/gemini-2.5-flash')

    summary_input = f"Overdue: {len(overdue_tasks)}, Next 7 Days: {len(upcoming_tasks)}, High Priority: {high_priority_tasks}"
    ai_response = model.generate_content(f"Write a 3-sentence project status summary: {summary_input}")
    ai_text = ai_response.text
except Exception as e:
    ai_text = f"AI Summary temporarily unavailable (Error: {e}). Please check your data tables below."

# --- FINAL OUTPUT ---
print("="*60)
print(f"MANAGEMENT TASK REPORT - GENERATED ON {today.date()}")
print("="*60)

print("\n[SECTION 1: OVERDUE TASKS]")
if not overdue_tasks:
    print("✅ No overdue tasks!")
else:
    for t in overdue_tasks:
        print(f"❌ {t['Task_Name']} (Owner: {t['Assigned_To']})")

print("\n[SECTION 2: UPCOMING TASKS (NEXT 7 DAYS)]")
if not upcoming_tasks:
    print("📅 No tasks due in the next 7 days.")
else:
    for t in upcoming_tasks:
        print(f"📅 {t['Task_Name']} (Due: {t['Due_Date']})")

print("\n[SECTION 3: PRIORITY REMINDER TABLE]")
print(f"{'PRIORITY':<10} | {'TASK NAMES'}")
print("-" * 60)
print(f"{'HIGH':<10} | {', '.join(high_priority_tasks) if high_priority_tasks else 'None'}")
print(f"{'MEDIUM':<10} | {', '.join(medium_priority_tasks) if medium_priority_tasks else 'None'}")
print(f"{'LOW':<10} | {', '.join(low_priority_tasks) if low_priority_tasks else 'None'}")

print("\n[SECTION 4: AI SUMMARY REPORT]")
print("-" * 60)
print(ai_text)
print("="*60)

MANAGEMENT TASK REPORT - GENERATED ON 2026-04-18

[SECTION 1: OVERDUE TASKS]
❌ Formulation Stability Test (Owner: Johana)
❌ Deliver samples to Customers (Owner: Johana)
❌ Update Safety Data Sheets (Owner: Michael)

[SECTION 2: UPCOMING TASKS (NEXT 7 DAYS)]
📅 Supplier Quality Audit (Due: 2026-04-20)
📅 Draft Pet Shampoo Label (Due: 2026-04-22)
📅 Weekly Team Sync (Due: 2026-04-23)
📅 Submit Python Project (Due: 2026-04-25)

[SECTION 3: PRIORITY REMINDER TABLE]
PRIORITY   | TASK NAMES
------------------------------------------------------------
HIGH       | Formulation Stability Test, Deliver samples to Customers, Submit Python Project, Update Safety Data Sheets
MEDIUM     | Supplier Quality Audit, Weekly Team Sync
LOW        | Draft Pet Shampoo Label

[SECTION 4: AI SUMMARY REPORT]
------------------------------------------------------------
The project currently has 3 overdue tasks that require immediate attention. Looking ahead, 4 tasks are scheduled for completion within the next seven 

**Problem # 4**

**Organizing Customer or Client Email Responses:**
In many customer-facing jobs, employees answer the same kinds of messages repeatedly, such as confirmations, reminders, shipping updates, or product questions. Writing similar replies again and again wastes time and can create inconsistent wording. This assignment asks students to generate standardized, personalized response drafts with Python.
**Objective**
Create a Python notebook that generates personalized email response drafts based on request type and customer details.
Suggested columns
**Outputs and deliverables**
• A generated response for each customer record
• One Colab notebook using conditionals and string formatting
• A table of customer names and draft messages
• At least three distinct message templates
• Grouped or categorized responses by request type
• A final output table of generated responses
• A short explanation of how template automation saves time





In [20]:
import pandas as pd
from google.colab import auth
import gspread
from google.auth import default

# --- STEP 1: AUTHENTICATION & DATA LOADING ---
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# This is the unique ID for your "Automating Customer Email Responses" sheet
spreadsheet_id = "14_04e6BPg8FSyiT4ywZmMLBqmFk0juqUGjnDFB2LauQ"

try:
    worksheet = gc.open_by_key(spreadsheet_id).sheet1
    customer_records = worksheet.get_all_records()
    print("✅ Connection Successful! Data loaded.")
except Exception as e:
    print(f"❌ Connection failed. Error: {e}")
    customer_records = []

# --- STEP 2: GENERATE RESPONSES (Deliverable: 3+ Templates) ---
draft_data = []
i = 0

while i < len(customer_records):
    c = customer_records[i]

    # We use .get() to avoid errors if a cell is empty
    name = c.get("Customer_Name", "Valued Customer")
    req = c.get("Request_Type", "General Inquiry")
    date = c.get("Reference_Date", "soon")

    # Logic for Templates
    if req == "Appointment Confirmation":
        msg = f"Hi {name}, your appointment is confirmed for {date}. We look forward to seeing you!"
    elif req == "Payment Reminder":
        msg = f"Hello {name}, this is a friendly reminder that your payment is due on {date}."
    elif req == "Shipping Update":
        msg = f"Good news {name}! Your order has shipped and is expected to arrive by {date}."
    else:
        msg = f"Hi {name}, thank you for reaching out regarding your {req}. We are reviewing your request."

    draft_data.append({
        "Customer Name": name,
        "Category": req,
        "Draft Message": msg
    })
    i += 1

# --- STEP 3: OUTPUT TABLE (Deliverable: Final Table) ---
if draft_data:
    final_table = pd.DataFrame(draft_data)
    print("\n" + "="*30)
    print("  FINAL GENERATED RESPONSES")
    print("="*30)
    display(final_table)
else:
    print("No records found to process.")

✅ Connection Successful! Data loaded.

  FINAL GENERATED RESPONSES


,Customer Name,Category,Draft Message
0,Milo Bernal,Appointment Confirmation,"Hi Milo Bernal, your appointment is confirmed ..."
1,Charlie Smith,Payment Reminder,"Hello Charlie Smith, this is a friendly remind..."
2,Bella Rodriguez,Shipping Update,Good news Bella Rodriguez! Your order has ship...
3,Luna Thompson,Payment Reminder,"Hello Luna Thompson, this is a friendly remind..."
4,Max Power,Appointment Confirmation,"Hi Max Power, your appointment is confirmed fo..."


**Problem # 5**

### **The Business Problem: "Inbox Exhaustion"**

### **What is the problem?**
The problem is **digital clutter**. Every day, our email inboxes are filled with "gray-mail"—newsletters, advertisements, and store updates that we once signed up for but no longer care about. Even though we stop reading them, we often just "Archive" or "Delete" them one by one to get them out of our sight. This is a repetitive, boring task that never ends because the same senders keep sending new emails every day.


### **Why does it matter?**
It matters because it wastes **time, focus, and money**.
* **Wasted Time:** Spending 10 minutes every morning cleaning out junk adds up to over an hour every week.
* **Missed Information:** When an inbox is messy, it is very easy to miss a truly important message—like an email from a professor about an MBA assignment or a message from a boss—because it is buried under 20 coupons.
* **Storage Costs:** Constant newsletters and large promotional emails with images quickly fill up the limited **15GB of free Google storage**. When memory runs out, you are forced to pay for extra monthly storage just to keep your account active. This automation helps prevent "storage bloat" by cutting off the source of the clutter.
* **Mental Clarity:** A crowded inbox creates a feeling of being overwhelmed before the workday even begins.


### **Who does it impact?**
This impacts **busy professionals and students** who receive a high volume of email. Specifically:
* **The MBA Student:** Who must keep track of fast-moving course deadlines.
* **The Chemical Formulator:** Who needs to see urgent lab results or supplier updates without distraction.
* **The "Budget-Conscious" User:** Anyone who wants to avoid paying for extra cloud storage by keeping their inbox lean and clean.

### **What would a solution look like?**
The solution is a **"Smart Assistant" program** written in Python that watches how you behave. Instead of you doing the work, the program follows these steps:

1.  **Observe:** It looks in your "Archive" folder to see which emails you moved away without opening.
2.  **Count:** It keeps a "Scoreboard." Every time you ignore a sender, it gives them a **Strike**.
3.  **Decide:** Once a sender hits **3 Strikes**, the program decides you are officially tired of them.
4.  **Report:** The program finds the "Unsubscribe" link for those senders and puts them all into one neat **Excel list**.

**The Result:** At the end of the week, you don't have to clean your inbox. You just open your list and click the links for the senders you want to remove. It turns a manual headache into an automated, organized system that protects both your time and your storage space.